<a target="_blank" href="https://colab.research.google.com/github/XPOZpublic/xpoz-cookbooks/blob/main/gemini/social-intelligence-with-xpoz-gemini.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Social Intelligence with XPOZ MCP + Gemini

This cookbook shows how to use [XPOZ](https://xpoz.ai) — a social-intelligence MCP server with **1.5 B+ indexed posts** across Twitter, Instagram, Reddit & TikTok — together with **Gemini** for real-time social media analysis.

**What you'll build:**
1. **Bitcoin Social Intelligence** — Pull tweets, Reddit threads & Instagram posts via XPOZ MCP, have Gemini analyse sentiment and themes, then render a styled HTML report.
2. **Bitcoin vs Ethereum** — Compare Bitcoin and Ethereum social presence — volume, sentiment, share of voice — and produce a competitive dashboard.

**Prerequisites — add these as Colab Secrets (🔑 icon in the sidebar):**
| Secret name | Where to get it |
|---|---|
| `XPOZ_API_KEY` | Free at [xpoz.ai](https://xpoz.ai) — 100 K results/month |
| `GOOGLE_API_KEY` | [aistudio.google.com](https://aistudio.google.com) |

## 1 · Setup

In [ ]:
!pip install -q "google-genai" httpx

In [ ]:
from google import genai
import json, re, csv, io, httpx, os
from datetime import datetime, timedelta
from IPython.display import HTML, display
from google.colab import userdata

# ── API keys from Colab Secrets ──────────────────────────────────────────
XPOZ_API_KEY = userdata.get("XPOZ_API_KEY")
GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")

# ── XPOZ MCP via direct HTTP (JSON-RPC) ─────────────────────────
XPOZ_URL = "https://mcp.xpoz.ai/mcp"
_msg_id = 0
_session_id = None

def _parse_sse(text):
    result = {}
    for line in text.split("\n"):
        if line.startswith("data: "):
            try:
                result = json.loads(line[6:])
            except json.JSONDecodeError:
                pass
    return result

async def _mcp_post(client, payload):
    global _session_id
    headers = {
        "Authorization": f"Bearer {XPOZ_API_KEY}",
        "Content-Type": "application/json",
        "Accept": "application/json, text/event-stream",
    }
    if _session_id:
        headers["Mcp-Session-Id"] = _session_id
    resp = await client.post(XPOZ_URL, json=payload, headers=headers)
    if "mcp-session-id" in resp.headers:
        _session_id = resp.headers["mcp-session-id"]
    ct = resp.headers.get("content-type", "")
    if "text/event-stream" in ct:
        return _parse_sse(resp.text)
    elif resp.text.strip():
        return resp.json()
    return {}

async def _ensure_session(client):
    global _session_id
    if _session_id:
        return
    await _mcp_post(client, {
        "jsonrpc": "2.0", "id": 0, "method": "initialize",
        "params": {"protocolVersion": "2025-03-26", "capabilities": {},
                   "clientInfo": {"name": "colab", "version": "1.0"}}
    })
    await _mcp_post(client, {
        "jsonrpc": "2.0", "method": "notifications/initialized"
    })

async def call_xpoz(tool_name, params):
    global _msg_id
    _msg_id += 1
    async with httpx.AsyncClient(timeout=120) as client:
        await _ensure_session(client)
        data = await _mcp_post(client, {
            "jsonrpc": "2.0", "id": _msg_id,
            "method": "tools/call",
            "params": {"name": tool_name, "arguments": params}
        })
    if "error" in data:
        print(f"  ⚠ MCP error: {data['error']}")
        return {"success": False, "data": {"results": []}}
    if "result" not in data:
        print(f"  ⚠ Unexpected response: {str(data)[:300]}")
        return {"success": False, "data": {"results": []}}
    content = data.get("result", {}).get("content", [{}])
    text = content[0].get("text", "{}") if content else "{}"
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return _parse_xpoz(text)

# ── Gemini client ─────────────────────────────────────────────────
gemini = genai.Client(api_key=GOOGLE_API_KEY)

# ── Helpers ──────────────────────────────────────────────────────
def days_ago(n: int) -> str:
    return (datetime.now() - timedelta(days=n)).strftime("%Y-%m-%d")

def _parse_xpoz(text: str) -> dict:
    """Parse XPOZ MCP compact tabular response into a Python dict."""
    result = {"success": "success: true" in text, "data": {}}
    hdr = re.search(r'results\[\d+\]\{([^}]+)\}:', text)
    if hdr:
        fields = hdr.group(1).split(',')
        rows = []
        for line in text[hdr.end():].split('\n'):
            if not line.startswith('    '):
                if rows or (line.strip() and not line.startswith(' ')):
                    break
                continue
            try:
                vals = next(csv.reader(io.StringIO(line.strip())))
                row = {}
                for i, f in enumerate(fields):
                    if i < len(vals):
                        v = vals[i].strip()
                        if v in ('null', ''):
                            row[f] = None
                        else:
                            try: row[f] = int(v)
                            except ValueError:
                                try: row[f] = float(v)
                                except ValueError: row[f] = v
                rows.append(row)
            except Exception:
                pass
        result["data"]["results"] = rows
    for m in re.finditer(r'^\s{2}(\w+):\s+(.+)$', text, re.MULTILINE):
        k, v = m.group(1), m.group(2).strip()
        if k.startswith('results'): continue
        try: result["data"][k] = int(v)
        except ValueError:
            try: result["data"][k] = float(v)
            except ValueError: result["data"][k] = v
    return result

def _num(v):
    """Safely convert to int for sorting/summing."""
    try: return int(v) if v else 0
    except (ValueError, TypeError): return 0

# ── Verify connection ──────────────────────────────────────────────
test = await call_xpoz("countTweets", {"phrase": "test", "startDate": days_ago(1)})
print(f"Connected to XPOZ MCP via HTTP — test count: {test['data'].get('count', 'ok')}")

---
## 2 · Bitcoin Social Intelligence Report

Fetch recent posts about **Bitcoin** from Twitter, Reddit and Instagram via XPOZ MCP, then ask Gemini to produce a structured sentiment analysis.

In [ ]:
TOPIC = "Bitcoin"

# ── Twitter ──────────────────────────────────────────────────────
count_r = await call_xpoz("countTweets", {
    "phrase": TOPIC, "startDate": days_ago(7)
})
tweet_count = count_r["data"].get("count", 0)

tw_r = await call_xpoz("getTwitterPostsByKeywords", {
    "query": "Bitcoin OR BTC", "startDate": days_ago(7), "limit": 300,
    "fields": ["text", "authorUsername", "likeCount", "retweetCount", "createdAtDate"]
})
tweets = tw_r["data"].get("results", [])

# ── Reddit ───────────────────────────────────────────────────────
rd_r = await call_xpoz("getRedditPostsByKeywords", {
    "query": "Bitcoin OR BTC", "startDate": days_ago(7), "limit": 200,
    "fields": ["title", "selftext", "authorUsername", "score", "subredditName", "createdAtDate"]
})
reddit_posts = rd_r["data"].get("results", [])

# ── Instagram ────────────────────────────────────────────────────
ig_r = await call_xpoz("getInstagramPostsByKeywords", {
    "query": "Bitcoin", "startDate": days_ago(7), "limit": 100,
    "fields": ["caption", "username", "likeCount", "commentCount", "createdAtDate"]
})
ig_posts = ig_r["data"].get("results", [])

print(f"Collected: {len(tweets)} tweets · {len(reddit_posts)} Reddit posts · {len(ig_posts)} Instagram posts")
print(f"Total tweet volume (7 days): {tweet_count:,}")

In [ ]:
# ── Build context for Gemini ─────────────────────────────────────────
tw_text = "\n".join(
    f"@{p.get('authorUsername','?')}: {p.get('text','')} "
    f"[likes:{p.get('likeCount',0)}, RTs:{p.get('retweetCount',0)}]"
    for p in tweets
)
rd_text = "\n".join(
    f"r/{p.get('subredditName','')} — {p.get('title','')}: "
    f"{(p.get('selftext') or '')[:200]} [score:{p.get('score',0)}]"
    for p in reddit_posts
)
ig_text = "\n".join(
    f"@{p.get('username','?')}: {(p.get('caption') or '')[:200]} "
    f"[likes:{p.get('likeCount',0)}, comments:{p.get('commentCount',0)}]"
    for p in ig_posts
)

prompt = f"""Analyse the social-media sentiment for {TOPIC} based on these posts from the last 7 days.
Total tweet volume: {tweet_count:,}

=== Twitter ({len(tweets)} posts) ===
{tw_text}

=== Reddit ({len(reddit_posts)} posts) ===
{rd_text}

=== Instagram ({len(ig_posts)} posts) ===
{ig_text}

Return your analysis in EXACTLY this format (keep the headers as-is):

OVERALL SENTIMENT: <Bullish/Bearish/Neutral> (<confidence>%)

KEY THEMES:
1. <theme> — <sentiment> — <one-line explanation>
2. …
3. …
4. …
5. …

RISK SIGNALS:
- <risk>
- …

OPPORTUNITIES:
- <opportunity>
- …

TOP POSTS:
1. <platform> @<author>: <quote> — <why it matters>
2. …
3. …"""

response = gemini.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt,
)
brand_analysis = response.text
print(brand_analysis)

In [ ]:
# ── Generate HTML report ─────────────────────────────────────────────
def _esc(t):
    t = t.replace('&','&amp;').replace('<','&lt;').replace('>','&gt;')
    t = re.sub(r'\*\*(.+?)\*\*', r'<strong>\1</strong>', t)
    return t.replace('\n\n','<br><br>').replace('\n','<br>')

rows_html = ""
for t in sorted(tweets, key=lambda x: _num(x.get('likeCount')), reverse=True)[:12]:
    rows_html += (
        f'<tr><td>@{t.get("authorUsername","?")}</td>'
        f'<td>{(t.get("text") or "")[:120]}</td>'
        f'<td class="r">{t.get("likeCount",0)}</td>'
        f'<td class="r">{t.get("retweetCount",0)}</td></tr>'
    )

total_engagement = (
    sum(_num(t.get('likeCount')) for t in tweets)
    + sum(_num(p.get('score')) for p in reddit_posts)
    + sum(_num(p.get('likeCount')) for p in ig_posts)
)

html = f"""<!DOCTYPE html>
<html lang="en"><head><meta charset="UTF-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>{TOPIC} Social Intelligence</title>
<style>
*{{margin:0;padding:0;box-sizing:border-box}}
body{{background:#0f172a;color:#e2e8f0;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;padding:2rem;max-width:1100px;margin:0 auto}}
h1{{font-size:2.2em;background:linear-gradient(135deg,#a855f7,#ec4899);-webkit-background-clip:text;-webkit-text-fill-color:transparent}}
h2{{color:#a855f7;margin:2rem 0 .8rem;font-size:1.15em;border-bottom:1px solid #1e293b;padding-bottom:.5rem}}
.hdr{{text-align:center;margin-bottom:2.5rem}}
.meta{{color:#64748b;font-size:.85em;margin-top:.5rem}}
.badge{{display:inline-block;background:#a855f720;color:#a855f7;padding:3px 12px;border-radius:20px;font-size:.82em;font-weight:600;margin:4px 2px}}
.grid{{display:grid;grid-template-columns:repeat(4,1fr);gap:14px;margin:1.2rem 0}}
.card{{background:#1e293b;border-radius:12px;padding:1.2rem;text-align:center}}
.card .v{{font-size:1.9em;font-weight:800;color:#a855f7}}
.card .l{{color:#64748b;font-size:.82em;margin-top:4px}}
.box{{background:#1e293b;border-radius:12px;padding:1.5rem;margin:.8rem 0;line-height:1.75;color:#cbd5e1}}
.box strong{{color:#e2e8f0}}
table{{width:100%;border-collapse:collapse;margin:.8rem 0}}
th,td{{padding:7px 10px;text-align:left;border-bottom:1px solid #1e293b;font-size:.85em}}
th{{color:#94a3b8;font-size:.75em;text-transform:uppercase}}
.r{{text-align:right}}
.ft{{text-align:center;color:#475569;font-size:.78em;margin-top:2.5rem;padding-top:1.2rem;border-top:1px solid #1e293b}}
.ft a{{color:#a855f7}}
</style></head><body>
<div class="hdr">
  <h1>{TOPIC} — Social Intelligence Report</h1>
  <div><span class="badge">XPOZ MCP</span> <span class="badge">Gemini</span> <span class="badge">Twitter</span> <span class="badge">Reddit</span> <span class="badge">Instagram</span></div>
  <div class="meta">Data via XPOZ MCP &middot; Analysis by Gemini &middot; {datetime.now().strftime('%Y-%m-%d %H:%M')}</div>
</div>
<div class="grid">
  <div class="card"><div class="v">{tweet_count:,}</div><div class="l">Tweets (7 d)</div></div>
  <div class="card"><div class="v">{len(tweets)}</div><div class="l">Twitter samples</div></div>
  <div class="card"><div class="v">{len(reddit_posts) + len(ig_posts)}</div><div class="l">Reddit + IG</div></div>
  <div class="card"><div class="v">{total_engagement:,}</div><div class="l">Total engagement</div></div>
</div>
<h2>Gemini Analysis</h2>
<div class="box">{_esc(brand_analysis)}</div>
<h2>Top Tweets by Engagement</h2>
<table><thead><tr><th>Author</th><th>Text</th><th class="r">Likes</th><th class="r">RTs</th></tr></thead>
<tbody>{rows_html}</tbody></table>
<div class="ft">Powered by <a href="https://xpoz.ai">XPOZ</a> Social Intelligence MCP &middot; Gemini</div>
</body></html>"""

with open("bitcoin-social-intelligence.html", "w") as f:
    f.write(html)

print("Saved: bitcoin-social-intelligence.html")
display(HTML(html))

---
## 3 · Bitcoin vs Ethereum

Compare **Bitcoin** and **Ethereum** social presence — tweet volume, sentiment, share of voice — and generate a competitive dashboard.

In [ ]:
COIN_A, COIN_B = "Bitcoin", "Ethereum"

async def fetch_coin(name, query):
    print(f"  Counting {name} tweets...")
    c = await call_xpoz("countTweets", {"phrase": name, "startDate": days_ago(7)})
    count = c["data"].get("count", 0)
    print(f"    → {count:,}")
    print(f"  Fetching {name} tweet samples...")
    t = await call_xpoz("getTwitterPostsByKeywords", {
        "query": query, "startDate": days_ago(7), "limit": 100,
        "fields": ["text", "authorUsername", "likeCount", "retweetCount"]
    })
    tweets = t["data"].get("results", [])
    print(f"    → {len(tweets)} tweets")
    print(f"  Fetching {name} Reddit posts...")
    r = await call_xpoz("getRedditPostsByKeywords", {
        "query": query, "startDate": days_ago(7), "limit": 50,
        "fields": ["title", "selftext", "score", "subredditName"]
    })
    reddit = r["data"].get("results", [])
    print(f"    → {len(reddit)} Reddit posts")
    return {
        "name": name, "tweet_count": count,
        "tweets": tweets, "reddit": reddit,
        "avg_likes": sum(_num(p.get("likeCount")) for p in tweets) / max(len(tweets), 1)
    }

print("Fetching Bitcoin data...")
data_btc = await fetch_coin("Bitcoin", "Bitcoin OR BTC")
print("\nFetching Ethereum data...")
data_eth = await fetch_coin("Ethereum", "Ethereum OR ETH")

total_vol = data_btc["tweet_count"] + data_eth["tweet_count"]
print(f"\nBitcoin: {data_btc['tweet_count']:,} tweets, avg {data_btc['avg_likes']:.0f} likes")
print(f"Ethereum: {data_eth['tweet_count']:,} tweets, avg {data_eth['avg_likes']:.0f} likes")
print(f"Share of Voice: BTC {data_btc['tweet_count']/max(total_vol,1)*100:.0f}% / ETH {data_eth['tweet_count']/max(total_vol,1)*100:.0f}%")

In [ ]:
btc_tweets_text = "\n".join(
    f"@{p.get('authorUsername','?')}: {p.get('text','')[:200]}"
    for p in data_btc["tweets"][:30]
)
eth_tweets_text = "\n".join(
    f"@{p.get('authorUsername','?')}: {p.get('text','')[:200]}"
    for p in data_eth["tweets"][:30]
)
btc_reddit_text = "\n".join(
    f"r/{p.get('subredditName','')}: {p.get('title','')} [score:{p.get('score',0)}]"
    for p in data_btc["reddit"][:15]
)
eth_reddit_text = "\n".join(
    f"r/{p.get('subredditName','')}: {p.get('title','')} [score:{p.get('score',0)}]"
    for p in data_eth["reddit"][:15]
)

prompt2 = f"""Compare Bitcoin and Ethereum social media presence over the last 7 days.

Bitcoin: {data_btc["tweet_count"]:,} tweets
Sample tweets:\n{btc_tweets_text}
Reddit:\n{btc_reddit_text}

Ethereum: {data_eth["tweet_count"]:,} tweets
Sample tweets:\n{eth_tweets_text}
Reddit:\n{eth_reddit_text}

Return your analysis in EXACTLY this format:

SHARE OF VOICE: BTC <X>% / ETH <Y>%

BITCOIN SENTIMENT: <Bullish/Bearish/Neutral> (<confidence>%)
ETHEREUM SENTIMENT: <Bullish/Bearish/Neutral> (<confidence>%)

KEY NARRATIVES:
- BTC: <top narrative>
- ETH: <top narrative>

COMPETITIVE DYNAMICS:
- <insight about how they compare>
- …

PREDICTION:
- <which coin has stronger social momentum and why>
"""

resp2 = gemini.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt2,
)
btc_eth_analysis = resp2.text
print(btc_eth_analysis)

In [ ]:
# ── Generate BTC vs ETH HTML dashboard ─────────────────────────
btc_pct = data_btc['tweet_count']/max(total_vol,1)*100
eth_pct = data_eth['tweet_count']/max(total_vol,1)*100

btc_rows = ""
for p in sorted(data_btc["tweets"], key=lambda x: _num(x.get("likeCount")), reverse=True)[:8]:
    btc_rows += (
        f'<tr><td>@{p.get("authorUsername","?")}</td>'
        f'<td>{(p.get("text") or "")[:120]}</td>'
        f'<td class="r">{_num(p.get("likeCount"))}</td></tr>'
    )
eth_rows = ""
for p in sorted(data_eth["tweets"], key=lambda x: _num(x.get("likeCount")), reverse=True)[:8]:
    eth_rows += (
        f'<tr><td>@{p.get("authorUsername","?")}</td>'
        f'<td>{(p.get("text") or "")[:120]}</td>'
        f'<td class="r">{_num(p.get("likeCount"))}</td></tr>'
    )

html2 = f"""<!DOCTYPE html>
<html lang="en"><head><meta charset="UTF-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Bitcoin vs Ethereum</title>
<style>
*{{{{margin:0;padding:0;box-sizing:border-box}}}}
body{{{{background:#0a0e1a;color:#e2e8f0;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;padding:2rem;max-width:1100px;margin:0 auto}}}}
h1{{font-size:2.2em;background:linear-gradient(135deg,#f59e0b,#8b5cf6);-webkit-background-clip:text;-webkit-text-fill-color:transparent}}
h2{{{{color:#f59e0b;margin:2rem 0 .8rem;font-size:1.15em;border-bottom:1px solid #1e293b;padding-bottom:.5rem}}}}
.hdr{{{{text-align:center;margin-bottom:2.5rem}}}}
.meta{{{{color:#64748b;font-size:.85em;margin-top:.5rem}}}}
.badge{{{{display:inline-block;padding:3px 12px;border-radius:20px;font-size:.82em;font-weight:600;margin:4px 2px}}}}
.b-orange{{{{background:#f59e0b20;color:#f59e0b}}}}
.b-purple{{{{background:#8b5cf620;color:#8b5cf6}}}}
.grid{{{{display:grid;grid-template-columns:repeat(4,1fr);gap:14px;margin:1.2rem 0}}}}
.card{{{{background:#1e293b;border-radius:12px;padding:1.2rem;text-align:center}}}}
.card .v{{{{font-size:1.9em;font-weight:800}}}}
.card .l{{{{color:#64748b;font-size:.82em;margin-top:4px}}}}
.orange{{{{color:#f59e0b}}}} .purple{{{{color:#8b5cf6}}}}
.bar-wrap{{{{background:#1e293b;border-radius:12px;padding:1.2rem;margin:1rem 0}}}}
.bar{{{{display:flex;height:36px;border-radius:8px;overflow:hidden;margin-top:.5rem}}}}
.bar-btc{{{{background:#f59e0b;display:flex;align-items:center;justify-content:center;font-weight:700;font-size:.85em}}}}
.bar-eth{{{{background:#8b5cf6;display:flex;align-items:center;justify-content:center;font-weight:700;font-size:.85em}}}}
.bar-label{{{{display:flex;justify-content:space-between;font-size:.8em;color:#94a3b8;margin-top:6px}}}}
.box{{{{background:#1e293b;border-radius:12px;padding:1.5rem;margin:.8rem 0;line-height:1.75;color:#cbd5e1}}}}
.box strong{{{{color:#e2e8f0}}}}
table{{{{width:100%;border-collapse:collapse;margin:.8rem 0}}}}
th,td{{{{padding:7px 10px;text-align:left;border-bottom:1px solid #1e293b;font-size:.85em}}}}
th{{{{color:#94a3b8;font-size:.75em;text-transform:uppercase}}}}
.r{{{{text-align:right}}}}
.ft{{{{text-align:center;color:#475569;font-size:.78em;margin-top:2.5rem;padding-top:1.2rem;border-top:1px solid #1e293b}}}}
.ft a{{{{color:#f59e0b}}}}
.cols{{{{display:grid;grid-template-columns:1fr 1fr;gap:16px}}}}
</style></head><body>
<div class="hdr">
  <h1>Bitcoin vs Ethereum</h1>
  <div><span class="badge b-orange">BTC: {data_btc["tweet_count"]:,} tweets</span> <span class="badge b-purple">ETH: {data_eth["tweet_count"]:,} tweets</span> <span class="badge b-orange">XPOZ MCP</span> <span class="badge b-orange">Gemini</span></div>
  <div class="meta">7-day window &middot; Data via XPOZ MCP &middot; Analysis by Gemini &middot; {datetime.now().strftime('%Y-%m-%d %H:%M')}</div>
</div>
<div class="grid">
  <div class="card"><div class="v orange">{data_btc["tweet_count"]:,}</div><div class="l">BTC Tweets</div></div>
  <div class="card"><div class="v purple">{data_eth["tweet_count"]:,}</div><div class="l">ETH Tweets</div></div>
  <div class="card"><div class="v orange">{btc_pct:.0f}%</div><div class="l">BTC Share</div></div>
  <div class="card"><div class="v purple">{eth_pct:.0f}%</div><div class="l">ETH Share</div></div>
</div>
<div class="bar-wrap">
  <strong>Share of Voice</strong>
  <div class="bar"><div class="bar-btc" style="width:{btc_pct:.0f}%">BTC {btc_pct:.0f}%</div><div class="bar-eth" style="width:{eth_pct:.0f}%">ETH {eth_pct:.0f}%</div></div>
  <div class="bar-label"><span>Bitcoin</span><span>Ethereum</span></div>
</div>
<h2>Gemini Analysis</h2>
<div class="box">{_esc(btc_eth_analysis)}</div>
<h2>Top Posts</h2>
<div class="cols">
<div>
<h3 style="color:#f59e0b;font-size:.95em;margin-bottom:.5rem">Bitcoin</h3>
<table><thead><tr><th>Author</th><th>Text</th><th class="r">Likes</th></tr></thead><tbody>{btc_rows}</tbody></table>
</div>
<div>
<h3 style="color:#8b5cf6;font-size:.95em;margin-bottom:.5rem">Ethereum</h3>
<table><thead><tr><th>Author</th><th>Text</th><th class="r">Likes</th></tr></thead><tbody>{eth_rows}</tbody></table>
</div>
</div>
<div class="ft">Powered by <a href="https://xpoz.ai">XPOZ</a> Social Intelligence MCP &middot; Gemini</div>
</body></html>"""

with open("btc-vs-eth.html", "w") as f:
    f.write(html2)

print("Saved: btc-vs-eth.html")
display(HTML(html2))

---
## Cleanup

In [ ]:
print("Done!")